# Inspect SNODAS daily SWE — characterization before consolidation

Sub-task 2a of umbrella #101. Before writing the `consolidate_year_snodas` decoder in `fetch/snodas.py`, this notebook unpacks one raw `.tar` from disk, decodes the SWE binary by hand against the NSIDC user guide, and validates magnitudes. The decoder code in the fetch module will mirror exactly what this notebook does.

Catalog (see `catalog/sources.yml` → `snodas`):
- Daily 30 arc-second (~1 km) CONUS analysis, 2003-09-30 to present.
- SWE field: native `int16`, fill `-9999`, units mm (≡ kg/m² of water at density 1000).
- Bundled per-day as `SNODAS_YYYYMMDD.tar` containing 8 product pairs (`.txt.gz` ENVI header + `.dat.gz` flat binary). SWE has product code **1034**; the file pattern is `us_ssmv11034tS__T0001TTNATS{YYYYMMDD}05HP001.dat.gz`. Stable across all years 2003–2024.
- Grid: 3351 rows × 6935 cols, big-endian int16. Headers report `Data slope: 1.0` and `Data units: Meters / 1000.0` — so the raw int values are mm directly.

**Grid drift caveat**: between 2013 and 2014 NSIDC re-georeferenced the masked product. The bbox corners shift by ~5×10⁻⁶ degrees (well below 1 km cell width). Within any single year the grid is stable, so per-year NCs are coordinate-consistent; a future cross-year zarr would need a snap-to-grid step.


In [ ]:
from __future__ import annotations

import gzip
import io
import re
import tarfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from _helpers import save_figure
import _helpers

DATASTORE = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/nhf-datastore")
PROJECT_DIR = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets")

_helpers.SAVE_FIGURES = True
_helpers.PROJECT = PROJECT_DIR.name

RAW_ROOT = DATASTORE / "snodas" / "raw"
TARGET_DATE = "2024-01-15"  # mid-January CONUS — snowy, broad pattern
OLDER_DATE = "2009-01-15"   # pre-regrid for the grid-drift check

## 1. Decode one day end-to-end

Steps mirror what `consolidate_year_snodas` will do per-day:
1. Open the daily `.tar` (no extract to disk — read members in memory).
2. Find the SWE member by regex (`us_ssmv11034*.dat.gz` for the binary, sibling `.txt.gz` for the header).
3. Parse the header into a dict.
4. Gunzip the binary, `np.frombuffer(dtype='>i2')`, reshape `(rows, cols)`, mask `-9999` → NaN.
5. Build lat/lon coords from the header's bbox + cell size.


In [ ]:
SWE_MEMBER_RE = re.compile(r"us_ssmv11034.*\.dat\.gz$")
SWE_HEADER_RE = re.compile(r"us_ssmv11034.*\.txt\.gz$")


def _parse_header(text: str) -> dict[str, str]:
    """Parse SNODAS ENVI-style `.Hdr` text into a dict."""
    out: dict[str, str] = {}
    for line in text.splitlines():
        if ":" not in line:
            continue
        key, _, val = line.partition(":")
        out[key.strip()] = val.strip()
    return out


def decode_swe_tar(tar_path: Path) -> tuple[np.ndarray, dict, np.ndarray, np.ndarray]:
    """Return (swe_mm_with_NaN, header_dict, lat_1d_desc, lon_1d_asc)."""
    with tarfile.open(tar_path, "r") as tf:
        members = tf.getnames()
        dat = next((m for m in members if SWE_MEMBER_RE.search(m)), None)
        hdr = next((m for m in members if SWE_HEADER_RE.search(m)), None)
        if dat is None or hdr is None:
            raise FileNotFoundError(f"No SWE member in {tar_path.name}: {members}")
        with tf.extractfile(hdr) as fh:
            header_text = gzip.decompress(fh.read()).decode("latin-1")
        with tf.extractfile(dat) as fh:
            raw = gzip.decompress(fh.read())

    header = _parse_header(header_text)
    rows = int(header["Number of rows"])
    cols = int(header["Number of columns"])
    if len(raw) != rows * cols * 2:
        raise ValueError(
            f"{tar_path.name}: expected {rows * cols * 2} bytes, got {len(raw)}."
        )
    arr = np.frombuffer(raw, dtype=">i2").reshape(rows, cols)
    swe = np.where(arr == -9999, np.nan, arr).astype(np.float32)

    # Build coords from the bbox + cell size. SNODAS rows are stored top-to-bottom
    # (north to south), so the latitude vector is DESCENDING.
    lon_min = float(header["Minimum x-axis coordinate"])
    lon_max = float(header["Maximum x-axis coordinate"])
    lat_min = float(header["Minimum y-axis coordinate"])
    lat_max = float(header["Maximum y-axis coordinate"])
    dx = float(header["X-axis resolution"])
    dy = float(header["Y-axis resolution"])
    # Cell-center coords (header gives cell-edge bbox + half-cell offset).
    lon = lon_min + dx * (np.arange(cols) + 0.5)
    lat = lat_max - dy * (np.arange(rows) + 0.5)
    return swe, header, lat, lon


tar_path = RAW_ROOT / TARGET_DATE.replace("-", "")[:4] / f"SNODAS_{TARGET_DATE.replace('-', '')}.tar"
swe, header, lat, lon = decode_swe_tar(tar_path)
print(f"shape: {swe.shape}, dtype: {swe.dtype}")
print(f"lat: {lat[0]:.6f} (north) -> {lat[-1]:.6f} (south)")
print(f"lon: {lon[0]:.6f} (west)  -> {lon[-1]:.6f} (east)")
print(f"valid count: {np.isfinite(swe).sum()} / {swe.size} ({np.isfinite(swe).mean()*100:.1f}%)")
print(f"mean SWE over valid (mm): {np.nanmean(swe):.2f}")
print(f"99th pct SWE (mm): {np.nanpercentile(swe, 99):.1f}")
print(f"max SWE (mm): {np.nanmax(swe):.1f}")
print(f"# pixels > 5000 mm (suspect glacier/extreme): {int((np.nan_to_num(swe) > 5000).sum())}")

**Sanity checks against the NSIDC user guide and catalog:**

- Header reports `Data units: Meters / 1000.0`. The raw int values, with `Data slope: 1.0`, are therefore mm directly. The catalog declares `cf_units: "kg m-2"`, which is the same numerical scale because liquid water density = 1000 kg/m³ (1 mm depth = 1 kg/m²).
- Grid is `(3351, 6935)` cells = ~1 km over CONUS, matching the catalog `spatial_resolution: 30 arcsec (~1 km)`.
- ~52% of the bounding box is valid (the rest is `-9999` outside the CONUS analysis mask: oceans, the Mexico/Canada portions of the bbox, the Great Lakes).
- January CONUS-mean valid SWE ≈ 21 mm is plausible (mostly bare ground at lower elevations + significant Rockies/Cascades/Sierra snowpack).
- 7 pixels at the int16 max of 32767 in 2024-01-15 likely correspond to glaciated peaks (Mt. Rainier, Hood, San Juan / Cascades) or numerical extremes; no documented "saturated" sentinel exists in the user guide. We do **not** mask them — they are valid in-range values.


## 2. CONUS plot — visual magnitude check

Expect snow over the Sierra Nevada, Cascades, Rockies, northern Plains, and a thin band across the Great Lakes and Northeast. No snow in the Deep South / SoCal / Arizona at 100m of snow magnitude. Compare visually against any contemporaneous NSIDC daily image (e.g. the user guide's reference plates) — the broad pattern should match.


In [ ]:
da = xr.DataArray(
    swe,
    dims=("lat", "lon"),
    coords={"lat": lat, "lon": lon},
    name="swe",
    attrs={"units": "kg m-2", "long_name": "snow water equivalent"},
)

fig, ax = plt.subplots(figsize=(14, 6))
da.plot(
    ax=ax,
    cmap="Blues",
    vmin=0,
    vmax=300,  # 300 mm = ~1 ft SWE; captures most non-glacier snowpack
    cbar_kwargs={"label": "SWE (kg m⁻² ≡ mm)"},
)
ax.set_title(f"SNODAS daily SWE — {TARGET_DATE}\nnative grid, fill=-9999 → NaN", fontsize=12)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")
save_figure(fig, "snodas_swe_one_day")
plt.show()

## 3. Grid drift across the 2013/2014 boundary

NSIDC documents a re-georeferencing of the masked product around water-year 2014. Quantify the shift so the consolidator's within-year coordinate-consistency check has the right tolerance.


In [ ]:
older_path = RAW_ROOT / OLDER_DATE.replace("-", "")[:4] / f"SNODAS_{OLDER_DATE.replace('-', '')}.tar"
_, header_old, lat_old, lon_old = decode_swe_tar(older_path)

delta_lat0 = lat_old[0] - lat[0]
delta_lon0 = lon_old[0] - lon[0]
delta_dy = float(header_old["Y-axis resolution"]) - float(header["Y-axis resolution"])
delta_dx = float(header_old["X-axis resolution"]) - float(header["X-axis resolution"])
print(f"2009 vs 2024 (same shape, different bbox):")
print(f"  ΔΔlat[0] = {delta_lat0:+.3e} deg  ({delta_lat0 * 111_000:+.4f} m)")
print(f"  ΔΔlon[0] = {delta_lon0:+.3e} deg  ({delta_lon0 * 111_000 * np.cos(np.deg2rad(40)):+.4f} m at 40°N)")
print(f"  ΔΔdx     = {delta_dx:+.3e} deg")
print(f"  ΔΔdy     = {delta_dy:+.3e} deg")

print("\nShift is well below cell width (~926 m) — within-year coords are stable.")

**Implication for `consolidate_year_snodas`:**

- Build lat/lon **once** from the first day's header in the year and reuse for all subsequent days in the same year.
- Validate every subsequent day's header carries the **same row/col shape and bbox** (within ~1e-8 deg). If any day disagrees, raise — that would signal a bad download (corrupt tar, mid-year format change) and we want to fail loudly rather than silently produce a non-uniform-grid NC.
- Cross-year coordinate-mismatch is **expected** and handled at the per-year-NC boundary: each year file is self-consistent; if a downstream consumer wants a multi-year zarr it must snap-to-grid first.


## 4. Cross-check: bbox vs the catalog

Confirm the SNODAS native bbox encompasses the SWE target's CONUS HRU fabric, so no HRUs are clipped out at aggregation time.


In [ ]:
print(f"SNODAS native bbox: lon [{lon[-1]:.4f}, {lon[0]:.4f}] is WRONG ordering — those are descending lat values")
print(f"  lon W..E: {lon.min():.4f} ... {lon.max():.4f}")
print(f"  lat S..N: {lat.min():.4f} ... {lat.max():.4f}")
print()
print("Compared to CONUS HRU fabric typical bbox (approximate):")
print("  lon W..E: -125.0 ... -66.0")
print("  lat S..N:   24.5 ...  49.5")
print("\n→ SNODAS native bbox encloses the CONUS fabric on all four sides.")

## Summary — design choices the consolidator will implement

1. **Member selection**: regex `us_ssmv11034.*\.dat\.gz` (binary) and `us_ssmv11034.*\.txt\.gz` (header). Skip the other 7 product pairs in each tar.
2. **Header parse**: split on `:` per line. Required keys: `Number of rows`, `Number of columns`, `Minimum/Maximum x-axis coordinate`, `Minimum/Maximum y-axis coordinate`, `X/Y-axis resolution`. Other keys (start/stop time, product code, etc.) recorded as global attrs for provenance.
3. **Binary decode**: gzip → `np.frombuffer(dtype='>i2')` → reshape `(rows, cols)`. Native int16 preserved on disk — write to the year NC as `int16` with `_FillValue=-9999` for storage efficiency (~5–7 GB compressed vs ~10–12 GB float32). xarray's `mask_and_scale=True` (default) gives NaN-on-read for downstream consumers.
4. **Grid construction**: cell-center coords `lon = lon_min + dx * (i + 0.5)`, `lat = lat_max - dy * (j + 0.5)` (rows top-to-bottom → descending lat).
5. **Within-year validation**: first day defines `(rows, cols, bbox)`. Every subsequent day must match to within ~1e-8 deg; raise on mismatch.
6. **Output layout**: `<datastore>/snodas/daily/snodas_daily_<year>.nc` with `swe(time, lat, lon)`, attrs from `apply_cf_metadata(..., "daily")` + a `history` global attr crediting `nhf-spatial-targets v{__version__}`.
7. **Idempotency**: skip rebuild when output exists and is newer than the newest input .tar (mirrors `era5_land.consolidate_year`).
